# Pauli propagation of OpenQASM circuits

The QDK compiles OpenQASM 2 and 3 to QIR, as it compiles Q#, and `paulipropagation-qdk` reads its circuits from QIR. This notebook goes from OpenQASM programs to expectation values on a hundred qubits.

`pauli_propagation.ipynb` starts from Q# and goes further into the method: gradients, error estimates and surrogates. All of it works on a circuit from OpenQASM in the same way.

The packages `qdk`, `numpy` and `paulipropagation-qdk` must be installed.

In [ ]:
import math
import time

import numpy as np
from qdk import code, openqasm, qsharp
from qdk.openqasm import ProgramType, import_openqasm
from qdk.simulation import NoiseConfig, run_qir

import paulipropagation_qdk as pp

## A circuit from OpenQASM

The program is the evolution of a chain of spins with an interaction between neighbours and a transverse field, in `layers` steps. `openqasm.compile` compiles it to QIR for the base profile, and `circuit_from_qir` reads the gates of the QIR program. Python writes the constants into the source.

In [ ]:
PROGRAM = """
OPENQASM 3.0;
include "stdgates.inc";

const int n = {n};
const int layers = {layers};
const float coupling = {coupling};
const float field = {field};

qubit[n] q;
output bit[n] c;

for int layer in [1:layers] {{
    for int i in [0:n - 2] {{
        rzz(coupling) q[i], q[i + 1];
    }}
    rx(field) q;
}}
c = measure q;
"""


def to_qir(n, layers, coupling=0.2, field=0.3):
    source = PROGRAM.format(n=n, layers=layers, coupling=coupling, field=field)
    return str(openqasm.compile(source, target_profile=qsharp.TargetProfile.Base))


qir = to_qir(12, 8)
circuit = pp.circuit_from_qir(qir)
circuit

## An expectation value

The observable is `Z` on one qubit in the middle of the chain. `expectation` propagates it through the circuit and returns its value in the state that the circuit prepares from all qubits in `|0>`. Qubit `i` of an observable is `q[i]` of the program.

In [ ]:
observable = pp.PauliSum(circuit.num_qubits)
observable.add("Z", [5])

exact = pp.expectation(circuit, observable, min_abs_coeff=0.0)
exact

Nothing was dropped here, so this is the exact value. On 12 qubits the QDK simulator can run the same QIR, and the average of the measurement results of `q[5]` over many shots must agree with it, up to the statistical error of the shots.

In [ ]:
shots = 20_000
results = run_qir(qir, shots=shots, seed=1)
sampled = sum(1 if shot[5] == qsharp.Result.Zero else -1 for shot in results) / shots
print(f"propagated {exact:.4f}   sampled {sampled:.4f} +- {1 / math.sqrt(shots):.4f}")

## A hundred qubits

The same program on 100 qubits is far beyond a state vector. The propagated observable grows too, and it is kept small by dropping terms: `max_weight` drops Pauli strings with more non-identity operators than the given number, and `min_abs_coeff` drops terms with small coefficients. The result is an approximation, and it converges as the truncation is loosened.

In [ ]:
circuit = pp.circuit_from_qir(to_qir(100, 20))
observable = pp.PauliSum(100)
observable.add("Z", [50])

for max_weight in (2, 4, 6, 8, 10):
    start = time.perf_counter()
    evolved = pp.propagate(circuit, observable, max_weight=max_weight, min_abs_coeff=1e-7)
    seconds = time.perf_counter() - start
    print(f"max_weight {max_weight:2}   {len(evolved):8} terms   <Z> = {evolved.overlap_with_zero():.8f}   {seconds:.2f} s")

## Noise

With a `NoiseConfig` of the QDK, the Pauli errors of every gate become channels of the circuit, and the propagation gives the expectation value of the noisy circuit. The simulators of the QDK estimate the same value from shots, which the first lines check on 12 qubits.

In [ ]:
def noise_of(strength):
    noise = NoiseConfig()
    noise.rzz.set_depolarizing(strength)
    noise.rx.set_depolarizing(strength / 10)
    return noise


small = pp.PauliSum(12)
small.add("Z", [5])
noisy = pp.expectation(pp.circuit_from_qir(qir, noise=noise_of(0.01)), small, min_abs_coeff=0.0)
results = run_qir(qir, shots=shots, noise=noise_of(0.01), seed=1)
sampled = sum(1 if shot[5] == qsharp.Result.Zero else -1 for shot in results) / shots
print(f"12 qubits: propagated {noisy:.4f}   sampled {sampled:.4f} +- {1 / math.sqrt(shots):.4f}")

for strength in (0.0, 0.002, 0.01):
    circuit = pp.circuit_from_qir(to_qir(100, 20), noise=noise_of(strength))
    evolved = pp.propagate(circuit, observable, max_weight=8, min_abs_coeff=1e-7)
    print(f"100 qubits, depolarizing {strength:5}   {len(evolved):8} terms   <Z> = {evolved.overlap_with_zero():.8f}")

## Gates that the QDK decomposes

`circuit_from_qir` reads Clifford gates, T gates, Toffoli gates and Pauli rotations. The QDK brings the rest of OpenQASM to these: controlled rotations, the modifiers `ctrl @`, `inv @` and `pow`, the general gate `U`, and the gates that a program defines.

In [ ]:
GATES = """
OPENQASM 3.0;
include "stdgates.inc";

gate entangle(theta) a, b {
    h a;
    cx a, b;
    rz(theta) b;
    cx a, b;
}

qubit[5] q;
h q;
for int i in [0:3] {
    entangle(0.3 * (i + 1)) q[i], q[i + 1];
}
cry(0.8) q[0], q[2];
ctrl @ rx(-0.4) q[4], q[1];
U(0.5, 1.1, -0.7) q[3];
ccx q[0], q[1], q[2];
inv @ t q[2];
pow(3) @ sx q[4];
cswap q[3], q[0], q[4];
"""

circuit = pp.circuit_from_qir(str(openqasm.compile(GATES, target_profile=qsharp.TargetProfile.Base)))
circuit

The QDK simulator runs the same source, and its state vector gives every expectation value directly.

In [ ]:
import_openqasm(GATES, name="Gates", program_type=ProgramType.Operation)
qsharp.eval("use qs = Qubit[5]; Gates(qs);")
# One axis for every qubit, in the order of the register.
state = np.array(qsharp.dump_machine().as_dense_state()).reshape((2,) * 5)
qsharp.eval("ResetAll(qs);")
probabilities = abs(state) ** 2

for qubit in range(5):
    observable = pp.PauliSum(5)
    observable.add("Z", [qubit])
    propagated = pp.expectation(circuit, observable, min_abs_coeff=0.0)
    one = probabilities.sum(axis=tuple(axis for axis in range(5) if axis != qubit))[1]
    print(f"<Z> on q[{qubit}]   propagated {propagated:+.12f}   state vector {1 - 2 * one:+.12f}")

## Parameters as inputs

A program can leave values open as `input` parameters. The QDK binds them when it compiles: the program is imported under a name once, and `compile` takes the values after it.

In [ ]:
PARAMETRIZED = """
OPENQASM 3.0;
include "stdgates.inc";

input float coupling;
input float field;

qubit[12] q;
output bit[12] c;

for int layer in [1:8] {
    for int i in [0:10] {
        rzz(coupling) q[i], q[i + 1];
    }
    rx(field) q;
}
c = measure q;
"""

qsharp.init(target_profile=qsharp.TargetProfile.Base)
import_openqasm(PARAMETRIZED, name="Evolve", program_type=ProgramType.File)

observable = pp.PauliSum(12)
observable.add("Z", [5])
for field in (0.1, 0.3, 0.5, 0.7):
    circuit = pp.circuit_from_qir(str(openqasm.compile(code.qasm_import.Evolve, 0.2, field)))
    print(f"field {field}   <Z> = {pp.expectation(circuit, observable, min_abs_coeff=0.0):.8f}")

The value at the field 0.3 is the one from the start of the notebook. For many values, `circuit.with_angles` and a `Surrogate` change the angles of a circuit without compiling again.

## OpenQASM 2

Programs in OpenQASM 2 with `qelib1.inc` are compiled in the same way. This one prepares a GHZ state, in which `XXXX` and `ZZ` on any two qubits have the value 1 and `Z` on one qubit has the value 0.

In [ ]:
GHZ = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[4];
creg c[4];
h q[0];
cx q[0],q[1];
cx q[1],q[2];
cx q[2],q[3];
measure q -> c;
"""

circuit = pp.circuit_from_qir(str(openqasm.compile(GHZ, target_profile=qsharp.TargetProfile.Base)))
for paulis, qubits in (("XXXX", [0, 1, 2, 3]), ("ZZ", [0, 3]), ("Z", [2])):
    observable = pp.PauliSum(4)
    observable.add(paulis, qubits)
    print(paulis, "on", qubits, "=", pp.expectation(circuit, observable))

## What a program has to be

| | |
| --- | --- |
| One circuit | Gates, then measurements. A program that branches on a measurement result is rejected. |
| Outputs | Declare the measured bits as `output`. Without it every classical variable is an output of the program, and the base profile has no numbers as outputs. |
| Inputs | Bound when the program is compiled, as above. |
| Qubits | Qubit `i` of an observable is `q[i]`. Several registers are numbered one after the other, in the order of their declarations. |
| Angles | The QDK reduces every angle to the range from 0 to 2π, so `circuit.angles` holds `2π - 0.4` for `rx(-0.4)`. Expectation values do not change. |